# 2. Missing data

Real lineage-tracing data has missing entries. HERACLES models two ways an
entry can go missing:

- **Heritable dropout** (`heritable_eps`): a site is lost in an ancestor,
  for example by silencing or a large deletion, and stays lost in all of its
  descendants.
- **Stochastic dropout** (`stochastic_delta`): a site is lost independently
  in each cell, for example because it was not captured during sequencing.

Both rates are learned from the data by default. This notebook fits two
simulated datasets that differ only in where the missing entries come from
and shows that the learned rates tell them apart. It takes about ten
minutes on a laptop CPU.

In [1]:
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from heracles import Heracles, robinson_foulds, sokal_collapse

# HERACLES logs every fit to stderr and to heracles.log in the output folder.
# Keep only warnings here so the notebook stays readable.
logging.getLogger("heracles").setLevel(logging.WARNING)
warnings.formatwarning = lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"

DATA = Path("data")
RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)
# Tutorial 1 explains this warning; hide it here.
warnings.filterwarnings("ignore", message="the loss was still improving")

`rf_to_truth` collapses a tree and the true tree the same way and returns
their Robinson-Foulds distance.

In [2]:
def rf_to_truth(tree, cm, truth_path):
    """RF distance (count, normalised) to the true tree, both Camin-Sokal collapsed."""
    truth = sokal_collapse(Path(truth_path).read_text(), cm)
    tree = sokal_collapse(tree, cm)
    return robinson_foulds(tree, truth), robinson_foulds(tree, truth, normalized=True)

## Two kinds of dropout

Both datasets have 250 cells and 30 sites with about a quarter of the
entries missing. In `cas9_heritable_dropout` every missing entry comes from
heritable silencing; in `cas9_stochastic_dropout` every missing entry is
independent per cell.

Heritable dropout leaves a trace: missing entries cluster in clades. A
quick look at the data shows this as many more cells that share the exact
pattern of missing sites.

In [3]:
data = {name: pd.read_csv(DATA / name / "characters.csv", index_col=0)
        for name in ["cas9_heritable_dropout", "cas9_stochastic_dropout"]}

for name, cm in data.items():
    patterns = (cm == -1).apply(lambda row: tuple(row), axis=1)
    shared = patterns.duplicated(keep=False) & (cm == -1).any(axis=1)
    print(f"{name}: {(cm == -1).to_numpy().mean():.0%} missing, "
          f"{shared.mean():.0%} of cells share their missing pattern with another cell")

cas9_heritable_dropout: 26% missing, 26% of cells share their missing pattern with another cell
cas9_stochastic_dropout: 26% missing, 0% of cells share their missing pattern with another cell


Fit both with the default settings, which learn both rates. The NJ column scores the RapidNJ tree each fit starts from (`model.diagnostics_["init_tree_newick"]`).

In [4]:
rows = []
for name, cm in data.items():
    model = Heracles(outdir=RUNS / name, seed=0)
    model.fit(cm)
    truth_path = DATA / name / "true_tree.nwk"
    rf, nrf = rf_to_truth(model.trees_["nj"], cm, truth_path)
    rf_nj, nrf_nj = rf_to_truth(model.diagnostics_["init_tree_newick"], cm, truth_path)
    rows.append({"dataset": name,
                 "heritable_eps_": round(model.heritable_eps_, 4),
                 "stochastic_delta_": round(model.stochastic_delta_, 4),
                 "HERACLES normalised RF": round(nrf, 3),
                 "NJ normalised RF": round(nrf_nj, 3)})
pd.DataFrame(rows).set_index("dataset")

,heritable_eps_,stochastic_delta_,HERACLES normalised RF,NJ normalised RF
dataset,,,,
cas9_heritable_dropout,0.2464,0.0072,0.108,0.377
cas9_stochastic_dropout,0.0047,0.2592,0.357,0.457


`heritable_eps_` is a rate per unit of branch length in the embedding and
`stochastic_delta_` is a probability per cell and site. Neither is in the
units the simulator used, so read them for which kind of dropout dominates
rather than as estimates of the simulated values.

If you know a rate, fix it instead of learning it, for example
`Heracles(heritable_eps=1e-6, learn_heritable_eps=False)`. A rate of exactly
0 cannot be learned, so pass a small positive value or turn learning off.

## Edit models

The edit model describes how the recorder writes and loses information. It
has two parts: an edit process (how sites get edited) and a missing-data
model (which sites go missing together).

| Model | Edit process | Missing data |
|---|---|---|
| `Cas9Model()` (default) | independent sites, each edited at most once | each site on its own |
| `PETracerModel(cassette_size=3)` | as Cas9 | whole cassettes of consecutive sites |
| `SequentialModel(cassette_size=K)` | sites in a cassette edited in order (DNA Typewriter) | whole cassettes |

In recorders such as PETracer, several target sites sit on one integrated
cassette, and losing the cassette blanks all of its sites in that cell at
once. The columns must then be ordered so that each cassette's sites are
next to each other:

```python
from heracles import PETracerModel
model = Heracles(outdir="runs/pet", edit_model=PETracerModel(cassette_size=3))
```

```bash
heracles fit barcodes.csv --outdir runs/pet --edit-model petracer --cassette-size 3
```

For `PETracerModel`, `stochastic_delta_` is the probability that a whole
cassette is lost in a cell. Tutorial 4 fits a sequential recorder.

A new recorder needs an `EditProcess` (how sites get edited), paired with a
missing-data model through `heracles.models.CustomModel`. The template and
the methods it must provide are documented in `heracles.models.base`.

**Next:** [3. Polishing the tree and ancestral states](03_polish_and_ancestral_states.ipynb).